## Load and Import Chunks

In [18]:
import json 
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

In [11]:
PROJECT_PATH = Path.cwd().parent

CHUNK_FILE = PROJECT_PATH / "data" / "processed" / "chunks" / "chunks.jsonl"

print(CHUNK_FILE)

/Users/pushkarkamat/Desktop/financial-rag/data/processed/chunks/chunks.jsonl


In [12]:
chunks = []

with open(CHUNK_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"Loaded Chunks: {len(chunks)}")

Loaded Chunks: 295


In [13]:
print("Number of chunks:", len(chunks))

if len(chunks) == 0:
    print("WARNING: chunks is empty!")
else:
    print("First chunk type:", type(chunks[0]))
    print("First chunk keys:", chunks[0].keys())

    for i, chunk in enumerate(chunks[:3]):
        print("=" * 80)
        print(f"CHUNK {i}")
        print("Text:")
        print(chunk["page_content"][:500])
        print("\nMetadata:")
        print(chunk["metadata"])

Number of chunks: 295
First chunk type: <class 'dict'>
First chunk keys: dict_keys(['page_content', 'metadata'])
CHUNK 0
Text:
NORTHSTAR FINANCIAL

Credit Risk Policy

CRP-001

Version 2.0 | Effective 1 April 2025

Classification: INTERNAL — CONTROLLED

Northstar Financial — Synthetic Internal Document. This document is fictional and created solely for educational and software-development purposes. It does not represent an actual financial institution, regulatory requirement, legal obligation, or financial advice.

Metadata:
{'document': '01_Credit_Risk_Policy_CRP-001_v2.0.docx', 'file_type': 'docx', 'page': None, 'sheet': None, 'row_start': None, 'row_end': None, 'content_type': 'text', 'section': None, 'image_path': None, 'source_path': '/Users/pushkarkamat/Desktop/financial-rag/data/raw/docx/01_Credit_Risk_Policy_CRP-001_v2.0.docx', 'table_id': None, 'chunk_id': 'chunk_00000000', 'char_count': 392}
CHUNK 1
Text:
Table 1
Section: Document Control

Row 1: Document ID: Document Title |

In [14]:
chunk['page_content']

'Table 2\nSection: Document Control > Approval Record\n\nRow 1: Role: Approving Authority | Name / Committee: Board Risk Committee | Decision: Approved | Date: 18 March 2025\nRow 2: Role: Document Owner | Name / Committee: Arvind Menon — Head of Credit Risk | Decision: Certified for issue | Date: 24 March 2025'

In [15]:
EMBED_MODEL = "BAAI/bge-large-en-v1.5"
embedding_model = SentenceTransformer(EMBED_MODEL)

print("Embedding dimension:", embedding_model.get_embedding_dimension())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding dimension: 1024


### Testing Embedding Model

In [17]:
test_text = chunk['page_content']

test_embedding = embedding_model.encode(test_text, normalize_embeddings=True)
print("Embedding Shape:", test_embedding.shape)
print("Value:", test_embedding[:5])

Embedding Shape: (1024,)
Value: [ 0.0225227  -0.01994173  0.0421556   0.01388161 -0.00375936]


## Embedding Chunks

In [19]:
texts = [chunk["page_content"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding Shape:", embeddings.shape)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Embedding Shape: (295, 1024)


## Save Embeddings

In [20]:
EMBEDDING_DIR = PROJECT_PATH / "data" / "processed" / "embeddings"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS_FILE = EMBEDDING_DIR / "embeddings.npy"

np.save(EMBEDDINGS_FILE, embeddings)

print(f"Saved embeddings: {embeddings.shape}")
print(f"Path: {EMBEDDINGS_FILE}")

Saved embeddings: (295, 1024)
Path: /Users/pushkarkamat/Desktop/financial-rag/data/processed/embeddings/embeddings.npy


## Validation 

In [21]:
saved_embeddings = np.load(EMBEDDINGS_FILE)
print("Reload Shape:", saved_embeddings.shape)

assert saved_embeddings.shape == embeddings.shape
assert np.allclose(saved_embeddings, embeddings)

print("Embedding file validation passed.")

Reload Shape: (295, 1024)
Embedding file validation passed.
